In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../')

In [2]:
import os
import time
import random
import warnings
import datetime
from pathlib import Path
from typing import Any, Callable, Optional, Union, cast

import torch
import torch.nn as nn

import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2
from torch.utils.data import DataLoader, default_collate

import numpy as np

%load_ext autoreload
%autoreload 2

from computer_vision.slowfast.data.dataset import UCF101, get_video_container
from computer_vision.slowfast.defaults import get_cfg

In [3]:
data_dirpath=Path('D:/data/ucf101')
root=data_dirpath/'UCF-101'
annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask'
output_dirpath=Path('D:/results/ucf101/slowfast/train')
metadata_path=data_dirpath/'metadata.pt'
class_id_path=annotation_path/"classInd.txt"

config='../config/SLOWFAST_4x16_R50.yaml'
split='train' # 'test', 'val'
# setup config see https://github.com/facebookresearch/SlowFast/blob/main/slowfast/utils/parser.py
cfg=get_cfg()
cfg.merge_from_file(config)
# train on https://github.com/facebookresearch/SlowFast/blob/main/tools/train_net.py#L493 
# construct_loader on https://github.com/facebookresearch/SlowFast/blob/main/slowfast/datasets/loader.py#L86
dataset_name=cfg.TRAIN.DATASET
batch_size=int(cfg.TRAIN.BATCH_SIZE/max(1, cfg.NUM_GPUS))
shuffle=True if split=='train' else False
drop_last=True if split=='train' else False

dataset=UCF101(cfg, mode=split, root=root, annotation_path=annotation_path, fold=1)

In [4]:
index=100

"""Given the video index, return the list of frames, label, and video index if the video 
can be fetched and decoded successfully; otherwise, repeatly find a random video that can be
decoded as a replacement
Args: 
    index (int): the video index provided by the pytorch sampler
Returns:
    frames (torch.Tensor): the frames sampled from the video. The dimension is 
        `channel` x `num_frames` x `height` x `width`
    label (int): the label of the current video
    index (int): index of the video or index of the replacement video 
"""
if dataset.mode in ['train', 'val']:
    # -1 indicates random sampling
    temporal_sample_index=-1 # clip idx
    spatial_sample_index=-1
    min_scale=dataset.cfg.DATA.TRAIN_JITTER_SCALES[0]
    max_scale=dataset.cfg.DATA.TRAIN_JITTER_SCALES[1]
    crop_size=dataset.cfg.DATA.TRAIN_CROP_SIZE
elif dataset.mode in ['test']:
    temporal_sample_index=( # clip idx
        dataset._spatial_temporal_idx[index]//dataset.cfg.TEST.NUM_SPATIAL_CROPS
    )
    # spatial_sample_index is \in [0,1,2], corresponding to left, center, or right if
    # width>height; otherwise, top, middle, or bottom if height > width
    spatial_sample_index=(
        (dataset._spatial_temporal_idx[index] % dataset.cfg.TEST.NUM_SPATIAL_CROPS)
        if dataset.cfg.TEST.NUM_SPATIAL_CROPS> 1
        else 1
    )
    min_scale, max_scale, crop_size=(
        [dataset.cfg.DATA.TEST_CROP_SIZE]*3 
        if dataset.cfg.TEST.NUM_SPATIAL_CROPS>1
        else [dataset.cfg.DATA.TRAIN_JITTER_SCALES[0]]**2 + \
        [dataset.cfg.DATA.TEST_CROP_SIZE]
    )
    # testing is deterministic and no jitter should be performed. 
    # min_scale, max_scale and crop_Size are expect to be the same
    assert len({min_scale, max_scale})==1
else: raise NotImplementedError(f"Does not support {dataset.mode}")

num_decode=(dataset.cfg.DATA.TRAIN_CROP_NUM_TEMPORAL if dataset.mode=='train' else 1)
min_scale, max_scale, crop_size=[min_scale], [max_scale], [crop_size]
if len(min_scale)<num_decode:
    min_scale+=[dataset.cfg.DATA.TRAIN_JITTER_SCALES[0]]*(num_decode-len(min_scale))
    max_scale+=[dataset.cfg.DATA.TRAIN_JITTER_SCALES[1]]*(num_decode-len(max_scale))
    crop_size+=(
        [dataset.cfg.MULTIGRID.DEFAULT_S]*(num_decode-len(crop_size))
        if dataset.cfg.MULTIGRID.LONG_CYCLE or dataset.cfg.MULTIGRID.SHORT_CYCLE
        else [dataset.cfg.DATA.TRAIN_CROP_SIZE]*(num_decode-len(crop_size))
    )
    assert dataset.mode in ['train', 'val']

# Try to decode and sample a clip from a video. If the video cannot be decoded, repeatedly find a
# random video replacement that can be decoded

In [5]:
for i_try in range(dataset._num_retries):
    video_container=None
    try:
        video_container=get_video_container(dataset._path_to_videos[index],
                                           dataset.cfg.DATA_LOADER.ENABLE_MULTI_THREAD_DECODE,
                                           dataset.cfg.DATA.DECODING_BACKEND)
    except Exception as e:
        print(f"Failed to load video from {dataset._path_to_videos[index]} with error {e}")
        if dataset.mode!='test': # try another one
            index=random.randint(0, len(dataset._path_to_videos)-1)
        continue # select a random video if the current video was not accessible
    if video_container is None:
        warnings.warn(f"Failed to load video_idx {index} from {dataset._path_to_videos[index]}; trail {i_try}")
        if dataset.mode!='test' and i_try>dataset._num_retries//8:
            # try again
            index=random.randint(0, len(dataset._path_to_videos)-1)
        continue

    frames_decoded, time_idx_decoded=([None]*num_decode, [None]*num_decode)
    num_frames=[dataset.cfg.DATA.NUM_FRAMES]
    sampling_rate=[dataset.cfg.DATA.SAMPLING_RATE]
    if len(num_frames)<num_decode: 
        num_frames.extend([num_frames[-1] for i in range(num_decode-len(num_frames))])
        # base case where keys have the same frame-reate as query
        sampling_rate.extend([sampling_rate[-1] for i in range(num_decode-len(sampling_rate))])
    elif len(num_frames)>num_decode:
        num_frames=num_frames[num_decode:]
        sampling_rate=sampling_rate[num_decode:]
    if dataset.mode=='train': assert len(min_scale)==len(max_scale)==len(crop_size)==num_decode
    target_fps=dataset.cfg.DATA.TARGET_FPS

    # Decode video. metadata is used to perform selective decoding
    
    break

In [6]:
# def decode
container=video_container
#sampling_rate
#num_frames,
clip_idx=temporal_sample_index
num_clips_uniform=dataset.cfg.TEST.NUM_ENSEMBLE_VIEWS
video_meta=(dataset._video_meta[index] if len(dataset._video_meta) <5e6 else {})
#target_fps=30,
backend=dataset.cfg.DATA.DECODING_BACKEND
max_spatial_scale=(min_scale[0] if all(x==min_scale[0] for x in min_scale) else 0)
use_offset=dataset.cfg.DATA.USE_OFFSET_SAMPLING
time_diff_prob=dataset.p_convert_dt if dataset.mode=='train' else 0.
gaussian_prob=0.0
min_delta=dataset.cfg.CONTRASTIVE.DELTA_CLIPS_MIN
max_delta=dataset.cfg.CONTRASTIVE.DELTA_CLIPS_MAX
temporally_rnd_clips=True
    